# Step 2: Histone feature matrix

Reads the 32 WIG tracks (8 marks x 4 stages) and computes mean signal density in a +/-1kb window around each gene's TSS. Missing files or empty windows are NaN, not 0 — no coverage isn't the same as no signal.

## Config

In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

WIG_DIR = RAW / "histone_wig"
GENE_COORDS = PROC / "gene_coordinates.csv"
OUT = PROC / "histone_features.csv"

MARKS = ["H3", "H3K4me1", "H3K4me3", "H3K9ac", "H3K18ac", "H3K27ac", "H4K5ac", "H4K8ac"]
STAGES = ["C1-8", "C9-13", "C14E", "C14L"]
WIG_CYCLE = {"C1-8": "c8", "C9-13": "c12", "C14E": "c14a", "C14L": "c14c"}
WINDOW_BP = 1000

assert GENE_COORDS.exists(), f"run 01_parse_gtf first — missing {GENE_COORDS}"
genes = pd.read_csv(GENE_COORDS)
print(f"{len(genes)} genes loaded")

16846 genes loaded


## WIG parser

In [7]:
# parses one WIG file into {chrom: (positions, values, span)} — span comes
# from the variableStep header line, not assumed

def parse_wig(wig_file):
    chrom_data = {}
    curr_chrom, curr_span = None, None
    positions, values = [], []

    def flush():
        if curr_chrom and positions:
            chrom_data[curr_chrom] = (np.array(positions, dtype=np.int64),
                                       np.array(values, dtype=np.float32),
                                       curr_span)

    with open(wig_file) as f:
        for line in f:
            if line.startswith("variableStep"):
                flush()
                chrom_m = re.search(r"chrom=(\S+)", line)
                span_m = re.search(r"span=(\d+)", line)
                curr_chrom = chrom_m.group(1) if chrom_m else None
                curr_span = int(span_m.group(1)) if span_m else 1  # 1 if no span given, per WIG spec
                positions, values = [], []
            elif curr_chrom and not line.startswith(("track", "#")):
                parts = line.split()
                if len(parts) >= 2:
                    positions.append(int(parts[0]))
                    values.append(float(parts[1]))
        flush()

    # register under both "chr2L" and "2L" so lookups work either way
    norm = {}
    for c, data in chrom_data.items():
        norm[c] = data
        norm[c if c.startswith("chr") else f"chr{c}"] = data
        norm[c.replace("chr", "")] = data
    return norm

## TSS windows

In [8]:
genes["tss"] = np.where(genes["strand"] == "-", genes["tx_end"], genes["tx_start"])
genes["win_start"] = genes["tss"] - WINDOW_BP
genes["win_end"] = genes["tss"] + WINDOW_BP

## Extract density per mark x stage

In [9]:
results = pd.DataFrame({"gene_name": genes["gene_name"]})
missing_report = {}

for mark in MARKS:
    for stage in STAGES:
        cycle = WIG_CYCLE[stage]
        col = f"{mark}_{stage}"
        wig_file = WIG_DIR / f"Dmel-{mark}-{cycle}.wig"

        if not wig_file.exists():
            print(f"missing file: {wig_file.name}")
            results[col] = np.nan
            missing_report[col] = len(genes)
            continue

        wig_map = parse_wig(wig_file)
        scores = np.full(len(genes), np.nan, dtype=np.float32)
        spans_seen = set()

        for chrom, grp in genes.groupby("chrom"):
            if chrom not in wig_map:
                continue
            pos, vals, span = wig_map[chrom]
            spans_seen.add(span)

            starts = np.searchsorted(pos, grp["win_start"].values, side="left")
            ends = np.searchsorted(pos, grp["win_end"].values, side="right")

            cum = np.concatenate([[0], np.cumsum(vals)])
            window_sum = cum[ends] - cum[starts]
            has_cov = ends > starts
            density = (window_sum * span) / (2 * WINDOW_BP)
            scores[grp.index[has_cov]] = density[has_cov]

        if len(spans_seen) > 1:
            print(f"WARNING {col}: span varies across chromosomes: {spans_seen}")

        results[col] = scores
        n_missing = np.isnan(scores).sum()
        missing_report[col] = n_missing
        print(f"{col}: span={spans_seen}, {n_missing}/{len(genes)} genes with no coverage")

results.to_csv(OUT, index=False)
print(f"\nsaved -> {OUT.relative_to(ROOT)}, {len(results)} genes x {len(results.columns)-1} features")

H3_C1-8: span={10}, 174/16846 genes with no coverage
H3_C9-13: span={10}, 172/16846 genes with no coverage
H3_C14E: span={10}, 145/16846 genes with no coverage
H3_C14L: span={10}, 144/16846 genes with no coverage
H3K4me1_C1-8: span={10}, 253/16846 genes with no coverage
H3K4me1_C9-13: span={10}, 152/16846 genes with no coverage
H3K4me1_C14E: span={10}, 153/16846 genes with no coverage
H3K4me1_C14L: span={10}, 141/16846 genes with no coverage
H3K4me3_C1-8: span={10}, 1165/16846 genes with no coverage
H3K4me3_C9-13: span={10}, 283/16846 genes with no coverage
H3K4me3_C14E: span={10}, 180/16846 genes with no coverage
H3K4me3_C14L: span={10}, 174/16846 genes with no coverage
H3K9ac_C1-8: span={10}, 263/16846 genes with no coverage
H3K9ac_C9-13: span={10}, 146/16846 genes with no coverage
H3K9ac_C14E: span={10}, 146/16846 genes with no coverage
H3K9ac_C14L: span={10}, 171/16846 genes with no coverage
H3K18ac_C1-8: span={10}, 1010/16846 genes with no coverage
H3K18ac_C9-13: span={10}, 155/16

## Missing-data summary

In [10]:
summary = pd.Series(missing_report).sort_values(ascending=False)
print("genes with no coverage, by feature:")
print(summary.to_string())

genes with no coverage, by feature:
H3K4me3_C1-8     1165
H3K18ac_C1-8     1010
H3K27ac_C1-8      650
H3K4me3_C9-13     283
H3K9ac_C1-8       263
H3K4me1_C1-8      253
H3K27ac_C9-13     196
H4K8ac_C1-8       188
H3K4me3_C14E      180
H3K27ac_C14L      178
H4K5ac_C14E       176
H3_C1-8           174
H3K4me3_C14L      174
H3_C9-13          172
H3K9ac_C14L       171
H3K18ac_C14E      160
H3K18ac_C9-13     155
H3K4me1_C14E      153
H3K27ac_C14E      152
H3K4me1_C9-13     152
H3K18ac_C14L      148
H4K5ac_C1-8       147
H3K9ac_C9-13      146
H3K9ac_C14E       146
H4K8ac_C14L       145
H4K5ac_C9-13      145
H3_C14E           145
H3_C14L           144
H4K5ac_C14L       144
H4K8ac_C9-13      142
H3K4me1_C14L      141
H4K8ac_C14E       140
